# Tutorial 5: MPS Basics

## Introduction

Machine Perception Services, or MPS, is a post-processing cloud service that we provide to Aria users.
It runs a set of proprietary, Spatial AI machine perception algorithms, that are designed for Project Aria glasses.
MPS is designed to provide superior accuracy and robustness compared to off-the-shelf open algorithms.

We are excited to share that we have extended MPS to Aria Gen2 users.
Currently, the supported MPS algorithms for Aria Gen2 include:
- **SLAM Single-Sequence Trajectory**: generates device trajectories, semi-dense point cloud data, and online calibration.
- **Hand Tracking**: generates 21 landmarks, wrist-to-device transforms, palm and wrist normals, and confidence scores.
- **Eye Gaze**: generates general and, when the wearer calibrated, personalized gaze.

**This tutorial is about the mechanics, not about any one algorithm.** It covers what an
MPS output folder contains, how to resolve paths into it, and the two ways to read it.
Consuming a specific algorithm's output is covered where that algorithm is covered:

| Algorithm | Tutorial |
| :-- | :-- |
| Trajectory, semi-dense point cloud, online calibration | `Tutorial_6_vio_and_trajectory` |
| Hand tracking | `Tutorial_7_hand_tracking` |
| Eye gaze | `Tutorial_8_eyetracking` |

Each of those covers the on-device source alongside the MPS one, because the useful
question is not "how do I read MPS" but "which source should I use, and how do they
differ".

**What you'll learn:**

- What an MPS output folder contains, and how the files are grouped
- How to resolve paths with `MpsDataPathsProvider` instead of hardcoding them
- How to query any MPS output by timestamp with `MpsDataProvider`
- How to check what actually ran, and at which version, before relying on it
- When to read the CSVs directly instead

**Prerequisites**
- Complete Tutorial 1 (VrsDataProvider Basics) to understand basic data provider concepts
- Download Aria Gen2 sample data: [VRS](https://www.projectaria.com/async/sample/download/?bucket=core&filename=aria_gen2_sample_data_1.vrs) and [MPS output zip file](https://www.projectaria.com/async/sample/download/?bucket=core&filename=aria_gen2_sample_data_1_mps_output_dec_2025.zip).

### ⚠️ Important Notes
- **Google Colab Users:**
  If you encounter a `ModuleNotFoundError: No module named 'rerun'` error after installing `rerun-sdk`, Colab may not recognize the new package until the runtime is restarted.
  **Fix:** Go to **Runtime → Restart session and run all**.

## Setup Environment (Google Colab)

If running on Google Colab, install projectaria-tools and download sample data (VRS file and MPS output).


In [ ]:
import sys
import os
import subprocess

google_colab_env = 'google.colab' in str(get_ipython())

if google_colab_env:
    print("Running from Google Colab, installing projectaria_tools and downloading sample data")

    # Install projectaria-tools
    !pip install projectaria-tools==2.3.0

    # Set up data path
    vrs_sample_path = "./vrs_sample_data"

    # Sample VRS file and MPS output URLs
    vrs_url = "https://www.projectaria.com/async/sample/download/?bucket=core&filename=aria_gen2_sample_data_1.vrs"
    mps_url = "https://www.projectaria.com/async/sample/download/?bucket=core&filename=aria_gen2_sample_data_1_mps_output_dec_2025.zip"

    vrs_filename = "aria_gen2_sample_data_1.vrs"
    mps_zip_filename = "aria_gen2_sample_data_1_mps_output_dec_2025.zip"

    vrs_file_path = os.path.join(vrs_sample_path, vrs_filename)
    mps_zip_path = os.path.join(vrs_sample_path, mps_zip_filename)
    mps_folder_path = os.path.join(vrs_sample_path, "mps_output")

    # Download and unzip commands
    command_list = [
        f"mkdir -p {vrs_sample_path}",
        f'curl -o {vrs_file_path} -C - -O -L "{vrs_url}"',
        f'curl -o {mps_zip_path} -C - -O -L "{mps_url}"',
        f"unzip -o {mps_zip_path} -d {mps_folder_path}"
    ]

    # Execute the commands for downloading dataset
    print(f"Downloading VRS and MPS sample data...")
    for command in command_list:
        !$command

    print(f"Download complete!")
    print(f"VRS file saved to: {vrs_file_path}")
    print(f"MPS data extracted to: {mps_folder_path}")

    # Running this command to trigger early failure of importing ReRun.
    # Should be resolved by restarting the Colab session.
    import rerun as rr
else:
    # For local environment, user needs to specify their own paths
    vrs_file_path = "path/to/your/recording.vrs"
    mps_folder_path = "path/to/your/mps/folder/"
    print(f"Please update vrs_file_path and mps_folder_path to point to your data")

In [ ]:
from projectaria_tools.core import data_provider, mps
import os

# Load local VRS file
vrs_data_provider = data_provider.create_vrs_data_provider(vrs_file_path)

## What is in an MPS output folder

MPS groups its output into one sub-folder per algorithm, each with its own summary:

```
mps_output/
├── slam/
│   ├── closed_loop_trajectory.csv
│   ├── open_loop_trajectory.csv
│   ├── semidense_points.csv.gz
│   ├── semidense_observations.csv.gz
│   ├── online_calibration.jsonl
│   └── summary.json
├── hand_tracking/
│   ├── hand_tracking_results.csv
│   └── summary.json
└── eye_gaze/
    ├── general_eye_gaze.csv
    ├── personalized_eye_gaze.csv      (only if the wearer calibrated)
    └── summary.json
```

Two things to know before you write a path by hand. Which sub-folders exist depends on
what you requested and on what the recording could support, so a missing folder is
normal rather than an error. And some names have changed over time -- older outputs
call the eye gaze files `generalized_eye_gaze.csv` and `calibrated_eye_gaze.csv`. The
path provider below handles both, which is the main reason to use it.

The `.csv.gz` files are read exactly like the `.csv` ones; compression is detected from
the extension.

In [ ]:
import os

print(f"MPS output folder: {mps_folder_path}\n")
for dirpath, dirnames, filenames in os.walk(mps_folder_path):
    depth = dirpath[len(str(mps_folder_path)):].count(os.sep)
    print(f"{'  ' * depth}{os.path.basename(dirpath) or mps_folder_path}/")
    for name in sorted(filenames):
        size_mb = os.path.getsize(os.path.join(dirpath, name)) / 1e6
        print(f"{'  ' * (depth + 1)}{name}  ({size_mb:.2f} MB)")

## Resolving paths: `MpsDataPathsProvider`

Point it at the output root and it hands back a resolved `MpsDataPaths`, with one group
per algorithm. An entry for an output that was not produced comes back as an empty
string, so checking for emptiness is the presence test.

In [ ]:
from projectaria_tools.core import mps

mps_data_paths = mps.MpsDataPathsProvider(mps_folder_path).get_data_paths()

print(f"root: {mps_data_paths.root}\n")

print("slam:")
for field in ("closed_loop_trajectory", "open_loop_trajectory", "semidense_points",
              "semidense_observations", "online_calibrations", "summary"):
    value = getattr(mps_data_paths.slam, field)
    print(f"  {field:24} {value or '(not present)'}")

print("\nhand_tracking:")
for field in ("hand_tracking_results", "summary"):
    value = getattr(mps_data_paths.hand_tracking, field, "")
    print(f"  {field:24} {value or '(not present)'}")

print("\neyegaze:")
for field in ("general_eyegaze", "personalized_eyegaze", "summary"):
    value = getattr(mps_data_paths.eyegaze, field)
    print(f"  {field:24} {value or '(not present)'}")

## Querying by timestamp: `MpsDataProvider`

`MpsDataProvider` wraps the resolved paths and answers timestamp queries across every
output, loading each file lazily on first use. This is what you want when you are
walking a VRS file and need the MPS result that lines up with each frame.

Two habits worth forming:

- **Check availability first.** Every getter has a matching `has_*`. The getters raise
  rather than return `None` when the underlying output is absent, so `has_*` is not
  optional politeness.
- **Check the version.** `get_slam_version()`, `get_hand_tracking_version()` and
  `get_eyegaze_version()` tell you which MPS release produced the output you are about
  to trust. Outputs from different releases are not always comparable.

In [ ]:
mps_data_provider = mps.MpsDataProvider(mps_data_paths)

print("what this output actually contains:")
for name in ("has_closed_loop_poses", "has_open_loop_poses", "has_online_calibrations",
             "has_semidense_point_cloud", "has_semidense_observations",
             "has_hand_tracking_results", "has_wrist_and_palm_poses",
             "has_general_eyegaze", "has_personalized_eyegaze"):
    print(f"  {name + '()':32} {getattr(mps_data_provider, name)()}")

print("\nversions:")
for name in ("get_slam_version", "get_hand_tracking_version", "get_eyegaze_version"):
    print(f"  {name + '()':32} {getattr(mps_data_provider, name)()}")

In [ ]:
from projectaria_tools.core import data_provider
from projectaria_tools.core.sensor_data import TimeQueryOptions

# Any device timestamp will do; an RGB frame time is the realistic case.
rgb_stream_id = vrs_data_provider.get_stream_id_from_label("camera-rgb")
probe_index = min(30, vrs_data_provider.get_num_data(rgb_stream_id) - 1)
probe_time_ns = vrs_data_provider.get_image_data_by_index(
    rgb_stream_id, probe_index
)[1].capture_timestamp_ns

print(f"querying every available MPS output at {probe_time_ns} ns\n")

if mps_data_provider.has_closed_loop_poses():
    pose = mps_data_provider.get_closed_loop_pose(probe_time_ns, TimeQueryOptions.CLOSEST)
    print(f"  closed loop pose:  translation {pose.transform_world_device.translation()[0]}")

if mps_data_provider.has_hand_tracking_results():
    hands = mps_data_provider.get_hand_tracking_result(probe_time_ns, TimeQueryOptions.CLOSEST)
    detected = [n for n, h in (("left", hands.left_hand), ("right", hands.right_hand)) if h]
    print(f"  hand tracking:     {detected or 'no hand detected at this instant'}")

if mps_data_provider.has_general_eyegaze():
    gaze = mps_data_provider.get_general_eyegaze(probe_time_ns, TimeQueryOptions.CLOSEST)
    print(f"  general eye gaze:  yaw {gaze.yaw:.4f} rad, pitch {gaze.pitch:.4f} rad")

if mps_data_provider.has_online_calibrations():
    online_calib = mps_data_provider.get_online_calibration(probe_time_ns, TimeQueryOptions.CLOSEST)
    print(f"  online calibration present: {online_calib is not None}")

## The other way: reading the files directly

`MpsDataProvider` is the right default, but it is a timestamp-query interface. When you
want the whole time series in memory -- to run statistics over a trajectory, or to plot
a full sequence -- read the file and get a plain list back:

| Output | Direct reader |
| :-- | :-- |
| Closed loop trajectory | `mps.read_closed_loop_trajectory(path)` |
| Open loop trajectory | `mps.read_open_loop_trajectory(path)` |
| Semi-dense point cloud | `mps.read_global_point_cloud(path)` |
| Point observations | `mps.read_point_observations(path)` |
| Online calibration | `mps.read_online_calibration(path)` |
| Hand tracking | `mps.hand_tracking.read_hand_tracking_results(path)` |
| Eye gaze | `mps.read_eyegaze(path)` |

Rule of thumb: walking a VRS file frame by frame, use `MpsDataProvider`; analysing a
whole sequence, use the readers. The algorithm tutorials use both, and say which and
why at each point.

---

## Related tutorials

- `Tutorial_1_vrs_data_provider_basics` — the VRS side of the same workflow
- `Tutorial_6_vio_and_trajectory` — MPS trajectory, point cloud and online calibration
- `Tutorial_7_hand_tracking` — MPS hand tracking
- `Tutorial_8_eyetracking` — MPS eye gaze, including general vs personalized